# CollateralAnalyzer Engine Test Suite

Comprehensive testing of the CollateralAnalyzer with real MCP blockchain data.
Each code block is kept to 5 lines or less for easy execution and debugging.

## Setup & Imports

In [ ]:
import sys
import asyncio
import json
from decimal import Decimal
from datetime import datetime

In [ ]:
sys.path.append('/home/mpo/algorand-showcase/algorand-lending-ecosystem/algorand-lending-business-logic')
from algorand_lending_bl.collateral import CollateralAnalyzer
from algorand_lending_bl.models import ASAToken, AssetType, LiquidityTier
from algorand_lending_bl.config import CollateralConfig

In [ ]:
import httpx
# MCP Service endpoints
READER_URL = "http://localhost:8002"
MARKET_DATA_URL = "http://localhost:8789"

## MCP Service Health Check

In [ ]:
async def check_mcp_health():
    try:
        async with httpx.AsyncClient() as client:
            reader_resp = await client.get(f"{READER_URL}/health")
            return reader_resp.status_code == 200
    except:
        return False

In [ ]:
health_status = await check_mcp_health()
print(f"🔍 MCP Services Status: {'✅ Healthy' if health_status else '❌ Offline'}")
if not health_status:
    print("💡 Start services: cd /home/mpo/algorand-showcase && ./start-real-services.sh")

## Test Asset Data Setup

In [ ]:
# Common Algorand assets for testing
algo_asset = ASAToken(
    asset_id=0,
    name="Algorand",
    symbol="ALGO"

In [ ]:
usdc_asset = ASAToken(
    asset_id=31566704,  # USDC on Algorand
    name="USD Coin",
    symbol="USDC",
    decimals=6

In [ ]:
# Sample test portfolio
test_assets = [algo_asset, usdc_asset]
test_quantities = [Decimal('100'), Decimal('500')]  # 100 ALGO, 500 USDC
loan_amount = Decimal('200')  # $200 USD loan request

## Initialize CollateralAnalyzer

In [ ]:
# Create analyzer with default config
analyzer = CollateralAnalyzer()
print(f"✅ CollateralAnalyzer initialized")
print(f"📊 Config: LTV={analyzer.config.max_ltv_ratio}, Min Ratio={analyzer.config.min_collateral_ratio}")

## Real Price Data Integration

In [ ]:
async def get_real_asset_price(symbol: str):
    """Get real-time price from MCP service"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.post(f"{MARKET_DATA_URL}/tools/get_price_feed", 
                                   json={"symbol": f"{symbol}/USD"})
            return resp.json().get("current_price", 0.0)
    except:
        return 0.0

In [ ]:
# Fetch real prices
algo_price = await get_real_asset_price("ALGO")
usdc_price = await get_real_asset_price("USDC")
print(f"💰 Real Prices: ALGO=${algo_price:.4f}, USDC=${usdc_price:.4f}")

## Basic Collateral Analysis Test

In [ ]:
# Run basic collateral analysis
analysis = await analyzer.analyze_collateral(
    assets=test_assets,
    quantities=test_quantities,
    loan_amount_usd=loan_amount
)

In [ ]:
print(f"📊 Collateral Analysis Results:")
print(f"   Total Value: ${analysis.total_collateral_value_usd:.2f}")
print(f"   LTV Ratio: {analysis.loan_to_value_ratio:.2%}")
print(f"   Risk Level: {analysis.overall_risk_level.value}")

## Portfolio Diversification Test

In [ ]:
# Test portfolio diversity
diversity = await analyzer.assess_portfolio_diversity(
    assets=test_assets,
    quantities=test_quantities
)

In [ ]:
print(f"🎯 Portfolio Diversity:")
print(f"   Diversity Score: {diversity.diversity_score:.2f}")
print(f"   Asset Count: {diversity.asset_count}")
print(f"   Concentration Risk: {diversity.concentration_risk.value}")

## Liquidation Scenario Testing

In [ ]:
# Test liquidation scenarios
scenarios = await analyzer.model_liquidation_scenarios(
    assets=test_assets,
    quantities=test_quantities,
    loan_amount_usd=loan_amount
)

In [ ]:
print(f"⚠️ Liquidation Scenarios:")
for scenario in scenarios[:3]:  # Show first 3 scenarios
    print(f"   {scenario.scenario_name}: Price drop {scenario.price_drop_percentage:.1%}")
    print(f"     Recovery: ${scenario.recovery_amount_usd:.2f}")

## Real Account Data Integration

In [ ]:
async def get_account_assets(address: str):
    """Get real account assets from blockchain"""
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.post(f"{READER_URL}/tools/get_account_info", 
                                   json={"address": address})
            return resp.json().get("account", {})
    except:
        return {}

In [ ]:
# Test with real account data
test_address = "7ZUECA7HFLZTXENRV24SHLU4AVPUTMTTDUFUBNBD64C5S3XM5THAIOF6Q"
account_data = await get_account_assets(test_address)
algo_balance = account_data.get("amount", 0) / 1_000_000  # Convert microAlgos
print(f"🏦 Real Account Balance: {algo_balance:.6f} ALGO")

## Risk Assessment with Real Data

In [ ]:
# Create real asset portfolio
real_assets = [algo_asset]
real_quantities = [Decimal(str(algo_balance))]
real_loan_amount = Decimal('50')  # $50 loan

In [ ]:
# Analyze real portfolio
real_analysis = await analyzer.analyze_collateral(
    assets=real_assets,
    quantities=real_quantities,
    loan_amount_usd=real_loan_amount
)

In [ ]:
print(f"🔍 Real Portfolio Analysis:")
print(f"   Portfolio Value: ${real_analysis.total_collateral_value_usd:.2f}")
print(f"   Loan Amount: ${real_loan_amount}")
print(f"   LTV Ratio: {real_analysis.loan_to_value_ratio:.2%}")
print(f"   Approved: {'✅ Yes' if real_analysis.loan_to_value_ratio <= analyzer.config.max_ltv_ratio else '❌ No'}")

## Stress Testing

In [ ]:
# Test edge cases
stress_tests = [
    {"name": "High LTV", "loan": Decimal('1000')},
    {"name": "Zero Collateral", "quantities": [Decimal('0')]},
    {"name": "Single Asset", "assets": [algo_asset]}
]

In [ ]:
print("🧪 Stress Test Results:")
for test in stress_tests:
    try:
        test_loan = test.get("loan", loan_amount)
        test_qtys = test.get("quantities", test_quantities)
        test_assets = test.get("assets", test_assets)
        result = await analyzer.analyze_collateral(test_assets, test_qtys, test_loan)
        print(f"   {test['name']}: ✅ LTV={result.loan_to_value_ratio:.2%}")
    except Exception as e:
        print(f"   {test['name']}: ❌ {str(e)[:50]}")

## Performance Benchmarking

In [ ]:
import time
# Benchmark analysis speed
start_time = time.time()
for _ in range(10):
    await analyzer.analyze_collateral(test_assets, test_quantities, loan_amount)
elapsed = time.time() - start_time

In [ ]:
print(f"⚡ Performance Benchmark:")
print(f"   10 analyses in {elapsed:.3f}s")
print(f"   Average: {elapsed/10:.3f}s per analysis")
print(f"   Rate: {10/elapsed:.1f} analyses/second")

## Test Results Summary

In [ ]:
# Comprehensive test summary
summary = {
    "timestamp": datetime.now().isoformat(),
    "mcp_services_healthy": health_status,
    "real_prices_fetched": algo_price > 0,
    "basic_analysis_completed": analysis is not None
}

In [ ]:
print("📋 Test Summary:")
for key, value in summary.items():
    status = "✅" if value else "❌"
    print(f"   {key}: {status} {value}")
print(f"\n🎯 CollateralAnalyzer Test Suite: {'PASSED' if all(summary.values()) else 'FAILED'}")